In [4]:
import gradio as gr
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Annotated

In [3]:
MODELS = [
    "gemma4:e4b",
    "ministral-3:8b",
    "llama3.1:8b",
]

In [12]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1",
)

In [16]:
class GetWeatherArgs(BaseModel):
    city: str = Field(description="City name, e.g 'Chicago'")

In [34]:
def get_weather(city: str) -> str:
    fake_data = {
        "chicago": "2℃ and snowy ❄️",
        "bishkek": "10℃ and rainy 🌦️",
        "london": "18℃ and cloudy ☁️",
        "tokyo": "12℃ and sunny ☀️",
        
    }
    return fake_data.get(city.lower(), "unknown city")

In [35]:
tools_json_get_weather = {
    "name": "get_weather",
    "description": "Get current weather for a city",
    "parameters": GetWeatherArgs.model_json_schema(),
}
tools = [{"type": "function", "function": tools_json_get_weather}]

In [36]:
messages = [
    {
        "role": "user",
        "content": "What is the weather in Tokyo"
    }
]

response = client.chat.completions.create(
    model=MODELS[0], 
    messages=messages,
    tools=tools
)
reply = response.choices[0].message

In [37]:
if reply.tool_calls:
    call = reply.tool_calls[0]
    print("Model wants to call: ", call.function.name, call.function.arguments)
    
    # Validate the arguments with Pydantic, then run your function
    args = GetWeatherArgs.model_validate_json(call.function.arguments)
    result = get_weather(args.city)
    print("Function returned: ", result)
    
    # Send the result back so the model can write the final answer
    messages.append(reply)
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,
        "content": result,
    })
    final = client.chat.completions.create(
        model=MODELS[1],
        messages=messages
    )
    final_result = final.choices[0].message.content
    print(f"Final answer with {MODELS[1]}: {final_result}")

else:
    print(reply.content)    
    

Model wants to call:  get_weather {"city":"Tokyo"}
Function returned:  12℃ and sunny ☀️
Final answer with ministral-3:8b: The current weather in Tokyo is **12°C with sunny skies** ☀️.
